# Evaluation of the recommender

Reproduces `scripts/evaluate.py` and looks at the queries that miss. Data: `data/postings.csv` (synthetic, see README) and `data/eval/`.

In [ ]:
import sys
sys.path.insert(0, '../src'); sys.path.insert(0, '../scripts')
import pandas as pd
from jobrec.data import load_postings, load_eval, load_judged
from jobrec.embed import TfidfEmbedder, GloveEmbedder, VECTORS_PATH
from jobrec.recommend import Recommender
from evaluate import evaluate, evaluate_judged, judged_agreement

postings = load_postings('../data/postings.csv')
queries, relevant = load_eval('../data/eval')
judged = load_judged('../data/eval')
len(postings), len(queries), sum(len(v) for v in judged.values())

## Precision@k and MRR per embedder and α

In [ ]:
rows = []
backends = [('tfidf', TfidfEmbedder)]
if VECTORS_PATH.exists():
    backends.append(('glove', GloveEmbedder))
for name, cls in backends:
    for alpha in (0.0, 0.5, 1.0):
        rec = Recommender(postings, cls(), alpha=alpha)
        res = evaluate(rec, queries, relevant)
        rows.append({'embedder': name, 'alpha': alpha, **res})
pd.DataFrame(rows)

## Which queries miss, and what they get instead

In [ ]:
rec = Recommender(postings, TfidfEmbedder(), alpha=0.5)
for q in queries.itertuples(index=False):
    top = rec.recommend(q.cv_text, 5)
    wrong = [(r.posting.id, r.posting.role_family, r.posting.title) for r in top
             if r.posting.id not in relevant[q.id]]
    if wrong:
        print(q.id, q.role_family, '->', wrong)

## Family rule vs hand judgement (10 judged queries, `data/eval/JUDGING.md`)

The same top-10 lists scored three ways: the family rule, the strict judgement (`relevant` only) and the lenient one (`relevant` or `arguable`). The judged subset was chosen to contain the cross-family CVs, so it is harder than the full set.

In [ ]:
jrows = []
for name, cls in backends:
    for alpha in (0.0, 0.5, 1.0):
        rec = Recommender(postings, cls(), alpha=alpha)
        jrows.append({'embedder': name, 'alpha': alpha,
                      **evaluate_judged(rec, queries, relevant, judged)})
pd.DataFrame(jrows)

In [ ]:
# Where the two label sets disagree: every disagreement is an 'arguable' pair.
pd.DataFrame(judged_agreement(relevant, judged)).fillna(0).astype(int)

In [ ]:
# The arguable pairs, by reason: seniority mismatches vs adjacent roles.
jd = pd.read_csv('../data/eval/judged.csv')
fam = dict(zip(queries.id, queries.role_family))
jd['same_family'] = [fam[q] == f for q, f in zip(jd.query_id, jd.posting_family)]
jd[jd.judgement == 'arguable'].groupby(['same_family', 'seniority']).size()

In [ ]:
# Q18 (data platform CV) at alpha = 0.5: the family rule calls 9 of 10 wrong,
# a human calls 8 of them arguable. Read the reasons.
rec = Recommender(postings, TfidfEmbedder(), alpha=0.5)
q18 = queries[queries.id == 'Q18'].iloc[0]
for r in rec.recommend(q18.cv_text, 10):
    row = jd[(jd.query_id == 'Q18') & (jd.posting_id == r.posting.id)].iloc[0]
    print(f'{r.posting.id} {row.judgement:9} {r.posting.title[:32]:32} | {row.reason}')

## One query in detail: matched and missing skills

In [ ]:
q = queries.iloc[6]  # Q07, the incubator CV
print(q.cv_text)
for r in rec.recommend(q.cv_text, 5):
    print(f'{r.score:.2f} sim={r.similarity:.2f} cov={r.skill_coverage:.2f} '
          f'{r.posting.id} {r.posting.title} | {r.explanation}')